In [37]:
#!pip install catboost

In [38]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OrdinalEncoder

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

sns.set_theme(style="whitegrid")

In [39]:
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(X.head())
print()

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Размер валидационной выборки: {X_val.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  

Размер обучающей выборки: 12384
Размер валидационной выборки: 4128
Размер тестовой выборки: 4128


In [40]:
class MyBoost:
    def __init__(self, n=400, lr=0.05, depth=7, subsample=1.0, colsample_bytree=1.0,
                 categorical_features=None, seed=42):
        """
        n : number of trees
        lr : learning rate
        depth : max depth of each tree
        subsample : fraction of training samples to use for each tree (0..1)
        colsample_bytree : fraction of features to use for each tree (0..1)
        categorical_features : list of column indices or names that are categorical
        seed : random seed
        """
        self.n = n
        self.lr = lr
        self.depth = depth
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.categorical_features = categorical_features
        self.seed = seed
        self.trees = []
        self.encoders = None
        self.feature_names_ = None

    def _preprocess_X(self, X, fit_encoder=False):
        """Convert categorical columns to numeric using OrdinalEncoder."""
        if self.categorical_features is None:
            return X.values if isinstance(X, pd.DataFrame) else X

        if isinstance(X, pd.DataFrame):
            cat_cols = [col for col in self.categorical_features if col in X.columns]
            num_cols = [col for col in X.columns if col not in cat_cols]
            X_cat = X[cat_cols].copy()
            X_num = X[num_cols].copy()
        else:
            cat_cols = self.categorical_features
            num_cols = [i for i in range(X.shape[1]) if i not in cat_cols]
            X_cat = X[:, cat_cols].copy()
            X_num = X[:, num_cols].copy()

        if fit_encoder:
            self.encoders = {}
            for col in cat_cols:
                enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
                X_cat_enc = enc.fit_transform(X_cat[[col]] if isinstance(X, pd.DataFrame) else X_cat[:, [i]])
                self.encoders[col] = enc

        X_cat_enc = np.zeros((X.shape[0], len(cat_cols)))
        for i, col in enumerate(cat_cols):
            enc = self.encoders[col]
            X_cat_enc[:, i] = enc.transform(X_cat[[col]] if isinstance(X, pd.DataFrame) else X_cat[:, [i]]).ravel()

        if isinstance(X, pd.DataFrame):
            X_num_vals = X_num.values
        else:
            X_num_vals = X_num
        X_processed = np.hstack([X_num_vals, X_cat_enc])

        if fit_encoder and self.feature_names_ is None:
            num_names = [f"{col}" for col in num_cols] if isinstance(X, pd.DataFrame) else [f"num_{i}" for i in range(X_num_vals.shape[1])]
            cat_names = [f"cat_{col}" for col in cat_cols] if isinstance(X, pd.DataFrame) else [f"cat_{i}" for i in range(len(cat_cols))]
            self.feature_names_ = num_names + cat_names
        return X_processed

    def fit(self, X, y):
        self.initial_leaf = y.mean()
        predictions = np.full(len(y), self.initial_leaf, dtype=float)

        X_processed = self._preprocess_X(X, fit_encoder=True)
        n_samples, n_features = X_processed.shape

        rng = np.random.RandomState(self.seed)

        for _ in range(self.n):
            antigrad = y - predictions

            if self.subsample < 1.0:
                n_subsample = int(n_samples * self.subsample)
                indices = rng.choice(n_samples, n_subsample, replace=False)
                X_sub = X_processed[indices]
                antigrad_sub = antigrad[indices]
            else:
                X_sub = X_processed
                antigrad_sub = antigrad

            if self.colsample_bytree < 1.0:
                n_cols = int(n_features * self.colsample_bytree)
                col_indices = rng.choice(n_features, n_cols, replace=False)
                X_sub = X_sub[:, col_indices]
            else:
                col_indices = None

            tree = DecisionTreeRegressor(max_depth=self.depth,
                                         random_state=self.seed,
                                         criterion="friedman_mse")
            tree.fit(X_sub, antigrad_sub)
            self.trees.append((tree, col_indices))

            if col_indices is not None:
                X_full_sub = X_processed[:, col_indices]
                predictions += self.lr * tree.predict(X_full_sub)
            else:
                predictions += self.lr * tree.predict(X_processed)

    def predict(self, X):
        X_processed = self._preprocess_X(X, fit_encoder=False)
        predictions = np.full(len(X_processed), self.initial_leaf, dtype=float)
        for tree, col_indices in self.trees:
            if col_indices is not None:
                X_sub = X_processed[:, col_indices]
                predictions += self.lr * tree.predict(X_sub)
            else:
                predictions += self.lr * tree.predict(X_processed)
        return predictions

    @property
    def feature_importances_(self):
        """Average of tree feature importances (weighted by learning rate is not needed for ranking)."""
        if not self.trees:
            return None
        n_features = None
        for tree, col_indices in self.trees:
            if col_indices is not None:
                n_cols = len(col_indices)
                if n_features is None:
                    pass
                pass
        if not hasattr(self, 'n_features_'):
            raise AttributeError("feature_importances_ available only after fit and if n_features_ is set.")
        total_imp = np.zeros(self.n_features_)
        count = 0
        for tree, col_indices in self.trees:
            if col_indices is not None:
                imp = tree.feature_importances_
                for i, idx in enumerate(col_indices):
                    total_imp[idx] += imp[i]
            else:
                total_imp += tree.feature_importances_
            count += 1
        return total_imp / count

In [41]:
class MyBoost:
    def __init__(self, n=400, lr=0.05, depth=7, subsample=1.0, colsample_bytree=1.0,
                 categorical_features=None, seed=42):
        self.n = n
        self.lr = lr
        self.depth = depth
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.categorical_features = categorical_features
        self.seed = seed
        self.trees = []
        self.encoders = None
        self.feature_names_ = None
        self.n_features_ = None

    def _preprocess_X(self, X, fit_encoder=False):
        if self.categorical_features is None:
            return X.values if isinstance(X, pd.DataFrame) else X

        if isinstance(X, pd.DataFrame):
            cat_cols = [col for col in self.categorical_features if col in X.columns]
            num_cols = [col for col in X.columns if col not in cat_cols]
            X_cat = X[cat_cols].copy()
            X_num = X[num_cols].copy()
        else:
            cat_cols = self.categorical_features
            num_cols = [i for i in range(X.shape[1]) if i not in cat_cols]
            X_cat = X[:, cat_cols].copy()
            X_num = X[:, num_cols].copy()

        if fit_encoder:
            self.encoders = {}
            for col in cat_cols:
                enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
                if isinstance(X, pd.DataFrame):
                    X_cat_enc = enc.fit_transform(X_cat[[col]])
                else:
                    X_cat_enc = enc.fit_transform(X_cat[:, [i]])
                self.encoders[col] = enc

        X_cat_enc = np.zeros((X.shape[0], len(cat_cols)))
        for i, col in enumerate(cat_cols):
            enc = self.encoders[col]
            if isinstance(X, pd.DataFrame):
                X_cat_enc[:, i] = enc.transform(X_cat[[col]]).ravel()
            else:
                X_cat_enc[:, i] = enc.transform(X_cat[:, [i]]).ravel()

        if isinstance(X, pd.DataFrame):
            X_num_vals = X_num.values
        else:
            X_num_vals = X_num

        X_processed = np.hstack([X_num_vals, X_cat_enc])

        if fit_encoder and self.feature_names_ is None:
            num_names = [f"{col}" for col in num_cols] if isinstance(X, pd.DataFrame) else [f"num_{i}" for i in range(X_num_vals.shape[1])]
            cat_names = [f"cat_{col}" for col in cat_cols] if isinstance(X, pd.DataFrame) else [f"cat_{i}" for i in range(len(cat_cols))]
            self.feature_names_ = num_names + cat_names
        return X_processed

    def fit(self, X, y):
        self.initial_leaf = y.mean()
        predictions = np.full(len(y), self.initial_leaf, dtype=float)

        X_processed = self._preprocess_X(X, fit_encoder=True)
        self.n_features_ = X_processed.shape[1]
        n_samples = X_processed.shape[0]

        rng = np.random.RandomState(self.seed)

        for _ in range(self.n):
            antigrad = y - predictions

            if self.subsample < 1.0:
                n_subsample = int(n_samples * self.subsample)
                indices = rng.choice(n_samples, n_subsample, replace=False)
                X_sub = X_processed[indices]
                antigrad_sub = antigrad[indices]
            else:
                X_sub = X_processed
                antigrad_sub = antigrad

            if self.colsample_bytree < 1.0:
                n_cols = int(self.n_features_ * self.colsample_bytree)
                col_indices = rng.choice(self.n_features_, n_cols, replace=False)
                X_sub_col = X_sub[:, col_indices]
            else:
                col_indices = None
                X_sub_col = X_sub

            tree = DecisionTreeRegressor(max_depth=self.depth,
                                         random_state=self.seed,
                                         criterion="friedman_mse")
            tree.fit(X_sub_col, antigrad_sub)
            self.trees.append((tree, col_indices))

            if col_indices is not None:
                X_full_sub = X_processed[:, col_indices]
                predictions += self.lr * tree.predict(X_full_sub)
            else:
                predictions += self.lr * tree.predict(X_processed)

    def predict(self, X):
        X_processed = self._preprocess_X(X, fit_encoder=False)
        predictions = np.full(len(X_processed), self.initial_leaf, dtype=float)
        for tree, col_indices in self.trees:
            if col_indices is not None:
                X_sub = X_processed[:, col_indices]
                predictions += self.lr * tree.predict(X_sub)
            else:
                predictions += self.lr * tree.predict(X_processed)
        return predictions

    @property
    def feature_importances_(self):
        if not self.trees or self.n_features_ is None:
            raise AttributeError("Model not fitted or n_features_ not set.")
        total_imp = np.zeros(self.n_features_)
        for tree, col_indices in self.trees:
            if col_indices is not None:
                imp = tree.feature_importances_
                for i, idx in enumerate(col_indices):
                    total_imp[idx] += imp[i]
            else:
                total_imp += tree.feature_importances_
        return total_imp / len(self.trees)

In [42]:
SELECTED_N_ESTIMATORS = 400

grid_xgb = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7],
    'subsample': [0.8, 1.0]
}

grid_lgb = {
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 127],
    'subsample': [0.8, 1.0]
}

grid_cat = {
    'learning_rate': [0.05, 0.1],
    'depth': [5, 7],
    'subsample': [0.8, 1.0]
}

def simple_grid_search(model_name, param_grid):
    print(f"--- Запуск Grid Search для {model_name} ---")
    best_rmse = float('inf')
    best_params = None

    start_time = time.time()

    for params in ParameterGrid(param_grid):
        if model_name == 'XGBoost':
            model = xgb.XGBRegressor(**params, n_estimators=SELECTED_N_ESTIMATORS, random_state=42, n_jobs=-1)
        elif model_name == 'LightGBM':
            model = lgb.LGBMRegressor(**params, n_estimators=SELECTED_N_ESTIMATORS, random_state=42, n_jobs=-1, verbose=-1)
        else:
            model = cb.CatBoostRegressor(**params, iterations=SELECTED_N_ESTIMATORS, random_seed=42, thread_count=-1, verbose=0)

        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))

        if rmse < best_rmse:
            best_rmse = rmse
            best_params = params

    total_time = time.time() - start_time
    print(f"Лучший Validation RMSE: {best_rmse:.4f}")
    print(f"Лучшие параметры: {best_params}\n")

    return best_params

best_params_dict = {}
best_params_dict['XGBoost'] = simple_grid_search('XGBoost', grid_xgb)
best_params_dict['LightGBM'] = simple_grid_search('LightGBM', grid_lgb)
best_params_dict['CatBoost'] = simple_grid_search('CatBoost', grid_cat)

--- Запуск Grid Search для XGBoost ---
Лучший Validation RMSE: 0.4621
Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 7, 'subsample': 0.8}

--- Запуск Grid Search для LightGBM ---
Лучший Validation RMSE: 0.4557
Лучшие параметры: {'learning_rate': 0.1, 'num_leaves': 31, 'subsample': 0.8}

--- Запуск Grid Search для CatBoost ---
Лучший Validation RMSE: 0.4564
Лучшие параметры: {'depth': 7, 'learning_rate': 0.1, 'subsample': 1.0}



In [43]:
from sklearn.model_selection import ParameterGrid

X_train_full = pd.concat([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

myboost_params = {
    'n': SELECTED_N_ESTIMATORS,
    'lr': 0.05,
    'depth': 7,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'categorical_features': None
}

results = []

for name in ['XGBoost', 'LightGBM', 'CatBoost', 'MyBoost']:
    if name == 'XGBoost':
        params = best_params_dict['XGBoost']
        model = xgb.XGBRegressor(n_estimators=SELECTED_N_ESTIMATORS, random_state=42, n_jobs=-1, **params)
    elif name == 'LightGBM':
        params = best_params_dict['LightGBM']
        model = lgb.LGBMRegressor(n_estimators=SELECTED_N_ESTIMATORS, random_state=42, n_jobs=-1, verbose=-1, **params)
    elif name == 'CatBoost':
        params = best_params_dict['CatBoost']
        model = cb.CatBoostRegressor(iterations=SELECTED_N_ESTIMATORS, random_seed=42, thread_count=-1, verbose=0, **params)
    else:
        model = MyBoost(**myboost_params)

    start_time = time.time()
    model.fit(X_train_full, y_train_full)
    train_time = time.time() - start_time

    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append({
        "Модель": name,
        "Время обучения": round(train_time, 3),
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2 Score": round(r2, 4)
    })

df_results = pd.DataFrame(results)
display(df_results)

,Модель,Время обучения,MAE,RMSE,R2 Score
0,XGBoost,2.396,0.2912,0.4476,0.8471
1,LightGBM,0.585,0.2862,0.4379,0.8537
2,CatBoost,2.400,0.2967,0.4493,0.8460
3,MyBoost,32.020,0.2925,0.4475,0.8472


In [44]:
model = MyBoost(**myboost_params)
model.fit(X_train_full, y_train_full)
importances = model.feature_importances_
if model.feature_names_ is not None:
    feat_imp = pd.DataFrame({'feature': model.feature_names_, 'importance': importances})
else:
    feat_imp = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False)
print("Feature importances from MyBoost:")
print(feat_imp)

Feature importances from MyBoost:
      feature  importance
0      MedInc    0.172328
7   Longitude    0.150365
6    Latitude    0.143950
5    AveOccup    0.137797
2    AveRooms    0.118618
3   AveBedrms    0.104442
4  Population    0.100506
1    HouseAge    0.071993


In [45]:
from sklearn.datasets import make_regression
X_cat, y_cat = make_regression(n_samples=500, n_features=3, random_state=42)
cat_col = np.random.choice(['A','B','C'], size=500)
X_cat_with_cat = np.column_stack([X_cat, cat_col])
df = pd.DataFrame(X_cat_with_cat, columns=['f1','f2','f3','cat'])
y_cat = y_cat

X_tr, X_te, y_tr, y_te = train_test_split(df, y_cat, test_size=0.2, random_state=42)

myboost_cat = MyBoost(n=100, lr=0.1, depth=5, subsample=0.9, colsample_bytree=0.9,
                      categorical_features=['cat'], seed=42)
myboost_cat.fit(X_tr, y_tr)
preds_cat = myboost_cat.predict(X_te)
print(f"Test RMSE with categorical feature: {np.sqrt(mean_squared_error(y_te, preds_cat)):.4f}")
print("Categorical support works without manual encoding.")

Test RMSE with categorical feature: 15.6585
Categorical support works without manual encoding.
